In [1]:
import pandas as pd
import numpy as np
import warnings

# --- Import Models ---
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import Ridge  # Meta-model

# --- Import Keras/TensorFlow for NN ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import metrics as k_metrics

# --- Import Sklearn Tools ---
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import (
    LabelEncoder, 
    OneHotEncoder, 
    StandardScaler, 
    OrdinalEncoder
)
from sklearn.metrics import accuracy_score 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# --- Suppress Warnings ---
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# --- 1. Define Constants & Seeds ---
N_SPLITS = 7
RANDOM_STATE = 42
SEED = 42 

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# --- 2. Load Data ---
print("Loading Data...")
try:
    # Competition Data
    train_df = pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv")
    test_df = pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv")
    
    # External Data (The "orig" dataset for your target mapping)
    orig_path = "/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv"
    orig_train_df = pd.read_csv(orig_path)
    
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")
    print(f"Original (External) shape: {orig_train_df.shape}")

except FileNotFoundError:
    print("Error: Data files not found. Please check paths.")
    exit()

# Store IDs
train_ids = train_df["id"]
test_ids = test_df["id"]

# --- 3. Feature Engineering (User Provided Logic) ---
print("Applying Advanced Feature Engineering...")

TARGET = 'loan_paid_back'

# 3.0 Prepare 'grade' column (needed for your logic)
# Assuming grade_subgrade is like "A1", "B2"
train_df['grade'] = train_df['grade_subgrade'].str[0]
test_df['grade'] = test_df['grade_subgrade'].str[0]
orig_train_df['grade'] = orig_train_df['grade_subgrade'].str[0]

# 3.1 Target Encoding using External Data (The "orig" logic)
# We map the mean target from the external dataset onto our train/test
# This avoids leakage because 'orig' is external!
BASE = [col for col in train_df.columns if col not in ['id', TARGET, 'grade', 'grade_subgrade']]
# Filter BASE to only cols that exist in orig
BASE = [c for c in BASE if c in orig_train_df.columns]

for col in BASE:
    # Mean encoding
    mean_map = orig_train_df.groupby(col)[TARGET].mean()
    train_df[f"orig_mean_{col}"] = train_df[col].map(mean_map)
    test_df[f"orig_mean_{col}"] = test_df[col].map(mean_map)
    
    # Count encoding
    count_map = orig_train_df.groupby(col).size()
    train_df[f"orig_count_{col}"] = train_df[col].map(count_map)
    test_df[f"orig_count_{col}"] = test_df[col].map(count_map)

# 3.2 Financial Ratios & Logic
# Applying to both Train and Test
for df in [train_df, test_df]:
    df['loan_to_income'] = df['loan_amount'] / (df['annual_income'] + 1)
    df['total_debt'] = df['debt_to_income_ratio'] * df['annual_income']
    df['available_income'] = df['annual_income'] * (1 - df['debt_to_income_ratio'])
    df['affordability'] = df['available_income'] / (df['loan_amount'] + 1)
    
    # Monthly payment estimation
    df['monthly_payment'] = df['loan_amount'] * (1 + df['interest_rate']/100) / 12
    df['payment_to_income'] = df['monthly_payment'] / (df['annual_income']/12 + 1)
    
    # Risk Score
    df['risk_score'] = (df['debt_to_income_ratio'] * 40 + 
                           (1 - df['credit_score']/850) * 30 + df['interest_rate'] * 2)
    
    # Grade Logic
    df['grade_number'] = df['grade_subgrade'].str[1].astype(int)
    grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
    df['grade_rank'] = df['grade'].map(grade_map)
    df['grade_combined'] = df['grade_rank'] * 10 + df['grade_number']
    
    # Interactions
    df['credit_interest'] = df['credit_score'] * df['interest_rate'] / 100
    df['income_credit'] = np.log1p(df['annual_income']) * df['credit_score'] / 1000
    df['debt_loan'] = df['debt_to_income_ratio'] * np.log1p(df['loan_amount'])
    
    # Log Transforms
    df['log_income'] = np.log1p(df['annual_income'])
    df['log_loan'] = np.log1p(df['loan_amount'])

# 3.3 Groupby Aggregations (Categorical Stats)
cat_cols = ['gender', 'marital_status', 'education_level', 
            'employment_status', 'loan_purpose', 'grade_subgrade']

for cat in cat_cols:
    # Note: Calculating means on TRAIN and mapping to TEST to prevent leakage
    mean_map_loan = train_df.groupby(cat)['loan_amount'].mean()
    train_df[f'{cat}_loan_mean'] = train_df[cat].map(mean_map_loan)
    test_df[f'{cat}_loan_mean'] = test_df[cat].map(mean_map_loan)
    
    mean_map_credit = train_df.groupby(cat)['credit_score'].mean()
    train_df[f'{cat}_credit_mean'] = train_df[cat].map(mean_map_credit)
    test_df[f'{cat}_credit_mean'] = test_df[cat].map(mean_map_credit)

# 3.4 Impute Missing Values (Created by mapping unseen categories)
# Your snippet logic:
num_cols = [col for col in train_df.columns if train_df[col].dtype in ['float64', 'int64'] 
            and col not in ['id', TARGET]]

for col in num_cols:
    if train_df[col].isnull().sum() > 0:
        median_val = train_df[col].median()
        train_df[col].fillna(median_val, inplace=True)
        test_df[col].fillna(median_val, inplace=True)

print("Feature Engineering Complete.")

# --- 4. Prepare X and y ---
# We Drop 'id' and 'TARGET'.
# NOTE: We KEEP the categorical columns because CatBoost/LGBM handle them well,
# but we also have all your new numerical engineered features.
X = train_df.drop([TARGET, 'id'], axis=1, errors='ignore')
y = train_df[TARGET]
X_test = test_df.drop(['id'], axis=1, errors='ignore')

# Align columns (ensure test has same cols as train)
X_test = X_test[X.columns]

# --- 5. Dynamic Feature Identification ---
# We need to update which columns are numerical vs categorical for the pipeline
ordinal_features_grade = ['grade_subgrade']
ordinal_features_employment = ['employment_status']

# All other object columns are OneHot
all_object_cols = X.select_dtypes(include='object').columns
one_hot_features = [
    col for col in all_object_cols 
    if col not in ordinal_features_grade and col not in ordinal_features_employment
]

# All numeric columns
numerical_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

print(f"Total Features: {X.shape[1]}")
print(f"Numerical: {len(numerical_features)}")
print(f"Categorical: {len(one_hot_features) + 2}")

# Convert target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
positive_class_label = le.classes_[1] 
positive_class_index = np.where(le.classes_ == positive_class_label)[0][0]

# --- 6. Preprocessing Pipelines ---
# Define Custom Orders
grades_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
subgrades_order = ['1', '2', '3', '4', '5']
grade_subgrade_order = [g + s for g in grades_order for s in subgrades_order]
employment_status_order = ['Employed', 'Self-employed', 'Retired', 'Student', 'Unemployed']

num_pipeline = Pipeline(steps=[('scaler', StandardScaler())])
cat_pipeline = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
grade_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[grade_subgrade_order], handle_unknown='use_encoded_value', unknown_value=-1))
])
employment_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[employment_status_order], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, one_hot_features),
        ('grade', grade_pipeline, ordinal_features_grade),
        ('employment', employment_pipeline, ordinal_features_employment)
    ],
    remainder='passthrough'
)

# --- 7. Define Models (Using your Parameters) ---

# 1. XGBoost
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 5,
    'learning_rate': 0.01, 'n_estimators': 10000, 'colsample_bytree': 0.8,
    'subsample': 0.85, 'min_child_weight': 3, 'gamma': 0.05,
    'reg_alpha': 0.05, 'reg_lambda': 1.0, 'random_state': RANDOM_STATE,
    'n_jobs': -1, 'device': 'cuda', 'tree_method': 'hist'
}
model_xgb = xgb.XGBClassifier(**xgb_params)

# 2. LightGBM
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'max_depth': 6, 'num_leaves': 50, 'learning_rate': 0.03,
    'n_estimators': 5000, 'colsample_bytree': 0.8, 'subsample': 0.8,
    'subsample_freq': 1, 'min_child_samples': 20, 'reg_alpha': 0.05,
    'reg_lambda': 0.1, 'random_state': RANDOM_STATE,
    'n_jobs': -1, 'device': 'gpu', 'verbose': -1
}
model_lgbm = lgb.LGBMClassifier(**lgb_params)

# 3. CatBoost
params_cat = {
    'iterations': 10000,
    'learning_rate': 0.01,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_strength': 1.0,
    'bagging_temperature': 0.5,
    'task_type': 'GPU',
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'verbose': False,
    'early_stopping_rounds': 200,
}
model_cat = CatBoostClassifier(**params_cat)

# 4. Neural Network
def create_nn_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(256), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(128), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.2),
        layers.Dense(64), layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001), 
                  loss='binary_crossentropy', metrics=[k_metrics.AUC(name='auc')])
    return model

# --- 8. Run CV ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds_xgb = np.zeros((len(X), 1))
oof_preds_lgbm = np.zeros((len(X), 1))
oof_preds_cat = np.zeros((len(X), 1))
oof_preds_nn = np.zeros((len(X), 1))

test_preds_xgb = np.zeros((len(X_test), 1))
test_preds_lgbm = np.zeros((len(X_test), 1))
test_preds_cat = np.zeros((len(X_test), 1))
test_preds_nn = np.zeros((len(X_test), 1))

print(f"\n--- Starting {N_SPLITS}-Fold CV for 4 Models ---")

for fold, (train_index, val_index) in enumerate(skf.split(X, y_encoded)):
    print(f"--- Fold {fold+1}/{N_SPLITS} ---")
    
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y_encoded[train_index], y_encoded[val_index]
    
    # Preprocessing
    preprocessor_fold = clone(preprocessor)
    X_train_fold_processed = preprocessor_fold.fit_transform(X_train_fold)
    X_val_fold_processed = preprocessor_fold.transform(X_val_fold)
    X_test_processed = preprocessor_fold.transform(X_test)

    # --- XGBoost ---
    print("Training XGB...")
    xgb_fold = clone(model_xgb)
    xgb_fold.set_params(early_stopping_rounds=200)
    xgb_fold.fit(X_train_fold_processed, y_train_fold, eval_set=[(X_val_fold_processed, y_val_fold)], verbose=False)
    oof_preds_xgb[val_index] = xgb_fold.predict_proba(X_val_fold_processed)[:, [positive_class_index]]
    test_preds_xgb += xgb_fold.predict_proba(X_test_processed)[:, [positive_class_index]] / N_SPLITS

    # --- LightGBM ---
    print("Training LGBM...")
    lgbm_fold = clone(model_lgbm)
    lgbm_fold.fit(X_train_fold_processed, y_train_fold, eval_set=[(X_val_fold_processed, y_val_fold)], callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_preds_lgbm[val_index] = lgbm_fold.predict_proba(X_val_fold_processed)[:, [positive_class_index]]
    test_preds_lgbm += lgbm_fold.predict_proba(X_test_processed)[:, [positive_class_index]] / N_SPLITS

    # --- CatBoost ---
    print("Training CatBoost...")
    cat_fold = clone(model_cat)
    cat_fold.fit(X_train_fold_processed, y_train_fold, eval_set=[(X_val_fold_processed, y_val_fold)], verbose=False)
    oof_preds_cat[val_index] = cat_fold.predict_proba(X_val_fold_processed)[:, [positive_class_index]]
    test_preds_cat += cat_fold.predict_proba(X_test_processed)[:, [positive_class_index]] / N_SPLITS

    # --- Neural Network ---
    print("Training NN...")
    input_dim = X_train_fold_processed.shape[1]
    nn_fold = create_nn_model(input_dim)
    nn_fold.fit(
        X_train_fold_processed, y_train_fold,
        validation_data=(X_val_fold_processed, y_val_fold),
        epochs=100, batch_size=256,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=15, mode='max', restore_best_weights=True)],
        verbose=0
    )
    oof_preds_nn[val_index] = nn_fold.predict(X_val_fold_processed, verbose=0)
    test_preds_nn += nn_fold.predict(X_test_processed, verbose=0) / N_SPLITS

# --- 9. Stacking ---
print("\n--- Individual Model Accuracy ---")
print(f"XGB: {accuracy_score(y_encoded, (oof_preds_xgb > 0.5).astype(int)):.5f}")
print(f"LGB: {accuracy_score(y_encoded, (oof_preds_lgbm > 0.5).astype(int)):.5f}")
print(f"CAT: {accuracy_score(y_encoded, (oof_preds_cat > 0.5).astype(int)):.5f}")
print(f"NN : {accuracy_score(y_encoded, (oof_preds_nn > 0.5).astype(int)):.5f}")

X_meta_train = np.hstack((oof_preds_xgb, oof_preds_lgbm, oof_preds_cat, oof_preds_nn))
X_meta_test = np.hstack((test_preds_xgb, test_preds_lgbm, test_preds_cat, test_preds_nn))

# Save Base Predictions
oof_df = pd.DataFrame({'oof_xgb': oof_preds_xgb.flatten(), 'oof_lgbm': oof_preds_lgbm.flatten(), 'oof_cat': oof_preds_cat.flatten(), 'oof_nn': oof_preds_nn.flatten(), 'target': y_encoded})
oof_df.to_csv("oof_predictions.csv", index=False)

test_preds_df = pd.DataFrame({'id': test_ids, 'test_xgb': test_preds_xgb.flatten(), 'test_lgbm': test_preds_lgbm.flatten(), 'test_cat': test_preds_cat.flatten(), 'test_nn': test_preds_nn.flatten()})
test_preds_df.to_csv("test_base_predictions.csv", index=False)

# Meta Model
print("\n--- Training Meta Model (Ridge) ---")
meta_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta_model.fit(X_meta_train, y_encoded)
meta_oof_preds = np.clip(meta_model.predict(X_meta_train), 0, 1)

best_acc = 0
best_threshold = 0
for threshold in np.arange(0.1, 0.9, 0.01):
    acc = accuracy_score(y_encoded, (meta_oof_preds > threshold).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print(f"Best OOF Accuracy: {best_acc:.6f} at threshold {best_threshold:.2f}")

# --- 10. Submission ---
final_predictions = np.clip(meta_model.predict(X_meta_test), 0, 1)
acc_str = f"{best_acc:.4f}".replace(".", "_")
submission_filename = f"submission_ridge_4stack_advanced_fe_acc_{acc_str}.csv"

submission_df = pd.DataFrame({"id": test_ids, TARGET: final_predictions})
submission_df.to_csv(submission_filename, index=False)
print(f"File saved: {submission_filename}")

2025-11-20 11:37:10.163997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763638630.390472      39 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763638630.452912      39 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading Data...
Train shape: (593994, 13)
Test shape: (254569, 12)
Original (External) shape: (20000, 22)
Applying Advanced Feature Engineering...
Feature Engineering Complete.
Total Features: 59
Numerical: 52
Categorical: 7

--- Starting 7-Fold CV for 4 Models ---
--- Fold 1/7 ---
Training XGB...
Training LGBM...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


...

Training NN...
--- Fold 6/7 ---
Training XGB...
Training LGBM...
Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


Training NN...
--- Fold 7/7 ---
Training XGB...
Training LGBM...
Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


Training NN...

--- Individual Model Accuracy ---
XGB: 0.90840
LGB: 0.90835
CAT: 0.90817
NN : 0.90654

--- Training Meta Model (Ridge) ---
Best OOF Accuracy: 0.908442 at threshold 0.50
File saved: submission_ridge_4stack_advanced_fe_acc_0_9084.csv


In [ ]:
# STACKED Model Best OOF F1:
# STACKED Model Best OOF Acuu: 0.908442
